# Suite VAL — Validators

`Validators.Validate(store, predicates)` checks each predicate on its matches among the objects given: every combination
of them whose schemas fit its symbols. It reports what does not hold, what is unknown and what raised, each match
labelled as mbse-schemas labels objects.

In [ ]:
import { Expressions as E } from "@mbse/expressions";
import { Constraints as C, Predicates as P, Validators as V } from "@mbse/patterns";
import { Errors } from "@mbse/schemas/Framework";
import { Contact, Directory, Phone, assert, book, listed, raises, same } from "./support.js";

const [the, c, p] = [E.variable("the"), E.variable("c"), E.variable("p")];
const owns = E.quantifier("any", "e", E.operation("entries", c, "phones"), E.variable("e").phone.eq(p));
const predicates = new C.OfSet.Builder().predicates(
  (b) => b.name("Adult").symbols({ the: Contact }).requires(the.age.ge(18n)),
  (b) => b.name("Phoned").symbols({ the: Contact }).requires(E.operation("count", E.operation("entries", the, "phones")).ge(1n)),
  (b) => b.name("OwnedNumbered").symbols({ c: Contact, p: Phone }).requires(E.operation("implies", owns, p.has("number"))),
).create();
const store = book();
const ann = store.Contact().name("ann").age(30n).phones((e: any) => e.phone((q: any) => q.number("555"))).create();
const bob = store.Contact().name("bob").phones((e: any) => e.phone((q: any) => q)).create();

## VAL-01 · A predicate is checked on each match

In [ ]:
{
  const validate = V.Validate(store, predicates);
  assert(validate(Contact, ann).length === 0);
  assert(same(validate(Contact, bob), ["the=Contact#0: 'Adult' is unknown"])); // bob's age is absent; he alone has no phone match
  const young = store.Contact().name("cy").age(9n).create();
  assert(same(validate(Contact, young), ["the=Contact#0: 'Adult' does not hold", "the=Contact#0: 'Phoned' does not hold"]));
  assert(same(validate(Contact, store.Phone().create()), ["the value is a 'Phone', not an instance of the given schema"]));
  assert(validate(Phone, store.Phone().create()).length === 0); // no predicate has a match of a phone alone
}

## VAL-02 · Reachable checks every match among what the root reaches: the cross product of the symbols

In [ ]:
{
  listed(store, ann, bob);
  const problems = V.Validate(store, predicates).Reachable(Directory, store.singleton("book.Directory"));
  assert(same(problems, ["the=Contact#2: 'Adult' is unknown",
    "c=Contact#2, p=Phone#4: 'OwnedNumbered' does not hold"]), problems.join("; ")); // bob's phone has no number
  assert(same(V.Validate(store, predicates).Reachable(Contact, ann), ["the=Contact#3: 'Adult' is unknown",
    "c=Contact#3, p=Phone#4: 'OwnedNumbered' does not hold"])); // from ann, through the directory, bob is reached too
}

## VAL-03 · An unknown result is reported, ignored or a violation

In [ ]:
{
  assert(V.Validate(store, predicates, { unknown: "ignore" })(Contact, bob).length === 0);
  assert(same(V.Validate(store, predicates, { unknown: "violation" })(Contact, bob), ["the=Contact#0: 'Adult' does not hold"]));
  assert(same(V.Validate(store, predicates, { unknown: "report" })(Contact, bob), ["the=Contact#0: 'Adult' is unknown"]));
  raises(Errors.ValueError, () => V.Validate(store, predicates, { unknown: "maybe" }),
    "unknown must be 'report', 'ignore' or 'violation', got 'maybe'");
  assert(same(V.UNKNOWN, ["report", "ignore", "violation"]));
}

## VAL-04 · A rule that raises is reported in place; a predicate that cannot be right is refused

In [ ]:
{
  const raising = new C.OfSet.Builder().predicates(
    (b) => b.name("Mixed").symbols({ the: Contact }).requires(E.operation("add", the.age, 1.5).ge(2.0)),
    (b) => b.name("Number").symbols({ the: Contact }).requires(the.age)).create();
  assert(same(V.Validate(store, raising)(Contact, ann), [
    "the=Contact#0: 'Mixed' raised TypeError: add expects numbers of one domain, got int and float",
    "the=Contact#0: 'Number' raised TypeError: a predicate must be a bool, got int"]));
  raises(Errors.ValueError, () => V.Validate(store, [(b: P.OfPredicate.Builder) => b.name("Older").symbols({ the: Contact })
    .parameters((p) => p.name("n")).requires(the.age.gt(E.variable("n")))]),
    "predicate 'Older' has parameters: it is checked where it is applied");
  raises(Errors.ValueError, () => V.Validate(store, [(b: P.OfPredicate.Builder) => b.name("X").symbols({ the: Contact }).requires(E.variable("n"))]),
    "predicate 'X': rule: variable 'n' is not bound");
}